In [ ]:
# ============================================
# END-TO-END: DeepLabv3+ Predict + Reconstruct (white-bg only)
#
# Reconstruction matches the ACTUAL patch-generation layout:
#   - Each original image is resized up to a multiple of patch_size (512)
#   - Patches are cut in row-major order: for y ... for x ...
#   - patch_idx increments continuously across the whole grid (NOT per row)
#
# So to reconstruct, for each base image we must know the grid's
# (rows, cols) — derived the SAME way the patch generator did, from the
# ORIGINAL image's dimensions. That's why `original_image_folder` below
# must point to the same folder used when patches were generated.
#
# Loads model -> predicts on each patch -> places each predicted patch at
# its correct (row, col) in a canvas -> resizes the canvas back down to the
# original image size -> saves ONLY the final reconstructed image.
# ============================================

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import numpy as np
import cv2
import os
import re
import math
import glob
from pathlib import Path
from collections import defaultdict

# ============================================
# 1. MODEL ARCHITECTURE
# ============================================

def depthwise_separable_conv(x, filters, kernel_size=3, strides=1, dilation_rate=1, name_prefix=''):
    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=strides,
        dilation_rate=dilation_rate,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_depthwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_depthwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_depthwise_relu')(x)

    x = Conv2D(
        filters,
        kernel_size=1,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_pointwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_pointwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_pointwise_relu')(x)
    return x

def atrous_spatial_pyramid_pooling(x, output_stride=16):
    b0 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='aspp0')(x)
    b0 = BatchNormalization(name='aspp0_bn')(b0)
    b0 = Activation('relu', name='aspp0_activation')(b0)

    if output_stride == 16:
        atrous_rates = (6, 12, 18)
    elif output_stride == 8:
        atrous_rates = (12, 24, 36)
    else:
        atrous_rates = (6, 12, 18)

    b1 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[0], use_bias=False, name='aspp1')(x)
    b1 = BatchNormalization(name='aspp1_bn')(b1)
    b1 = Activation('relu', name='aspp1_activation')(b1)

    b2 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[1], use_bias=False, name='aspp2')(x)
    b2 = BatchNormalization(name='aspp2_bn')(b2)
    b2 = Activation('relu', name='aspp2_activation')(b2)

    b3 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[2], use_bias=False, name='aspp3')(x)
    b3 = BatchNormalization(name='aspp3_bn')(b3)
    b3 = Activation('relu', name='aspp3_activation')(b3)

    b4 = GlobalAveragePooling2D()(x)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='image_pooling')(b4)
    b4 = BatchNormalization(name='image_pooling_bn')(b4)
    b4 = Activation('relu', name='image_pooling_activation')(b4)

    def resize_to_feature_map(inputs):
        feature_map, pooled_features = inputs
        shape = tf.shape(feature_map)
        target_height = shape[1]
        target_width = shape[2]
        return tf.image.resize(pooled_features, [target_height, target_width])

    b4 = tf.keras.layers.Lambda(resize_to_feature_map)([x, b4])

    x = layers.Concatenate()([b0, b1, b2, b3, b4])

    x = Conv2D(256, (1, 1), padding='same', use_bias=False, name='concat_projection')(x)
    x = BatchNormalization(name='concat_projection_bn')(x)
    x = Activation('relu', name='concat_projection_activation')(x)
    x = Dropout(0.1)(x)

    return x

def deeplabv3plus_model(input_size=(512, 512, 3), output_stride=16):
    inputs = Input(input_size)

    # Entry flow
    x = Conv2D(32, (3, 3), strides=2, padding='same', use_bias=False, name='entry_flow_conv1_1')(inputs)
    x = BatchNormalization(name='entry_flow_conv1_1_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_1_relu')(x)

    x = Conv2D(64, (3, 3), padding='same', use_bias=False, name='entry_flow_conv1_2')(x)
    x = BatchNormalization(name='entry_flow_conv1_2_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_2_relu')(x)

    # Block 1
    residual = Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_1')
    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Block 2
    residual = Conv2D(256, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_1')
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    low_level_features = x

    # Block 3
    residual = Conv2D(728, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_1')
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Middle flow
    for i in range(8):
        residual = x
        prefix = f'middle_flow_unit_{i + 1}'
        x = Activation('relu')(x)
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_1')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_2')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_3')
        x = layers.Add()([x, residual])

    # Exit flow
    residual = Conv2D(1024, (1, 1), strides=1, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='exit_flow_block1_1')
    x = depthwise_separable_conv(x, 1024, name_prefix='exit_flow_block1_2')
    x = layers.Add()([x, residual])

    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_1')
    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_2')
    x = depthwise_separable_conv(x, 2048, name_prefix='exit_flow_block2_3')

    # ASPP
    x = atrous_spatial_pyramid_pooling(x, output_stride)

    # Decoder
    low_level_features = Conv2D(48, (1, 1), padding='same', use_bias=False, name='feature_projection0')(low_level_features)
    low_level_features = BatchNormalization(name='feature_projection0_bn')(low_level_features)
    low_level_features = Activation('relu', name='feature_projection0_relu')(low_level_features)

    def upsample_to_match(inputs):
        encoder_features, low_level_features = inputs
        low_level_shape = tf.shape(low_level_features)
        target_height = low_level_shape[1]
        target_width = low_level_shape[2]
        return tf.image.resize(encoder_features, [target_height, target_width])

    x = tf.keras.layers.Lambda(upsample_to_match)([x, low_level_features])
    x = layers.Concatenate()([x, low_level_features])

    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv0')
    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv1')

    def upsample_to_input_size(x):
        return tf.image.resize(x, [512, 512])

    x = tf.keras.layers.Lambda(upsample_to_input_size)(x)
    outputs = Conv2D(1, kernel_size=1, activation='sigmoid', name='output')(x)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

# ============================================
# 2. LOSS FUNCTIONS & METRICS (needed to compile/load)
# ============================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def combined_focal_dice_loss(y_true, y_pred):
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    alpha = 0.8
    gamma = 2.0
    alpha_t = y_true * alpha + (K.ones_like(y_true) - y_true) * (1 - alpha)
    p_t = y_true * y_pred + (K.ones_like(y_true) - y_true) * (K.ones_like(y_true) - y_pred)
    focal_loss_val = -alpha_t * K.pow((K.ones_like(y_true) - p_t), gamma) * K.log(p_t)
    focal = K.mean(focal_loss_val)
    dice = dice_loss(y_true, y_pred)
    return focal + dice

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def binary_accuracy(y_true, y_pred):
    return K.mean(K.equal(y_true, K.round(y_pred)))

def precision(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    return true_positives / (predicted_positives + K.epsilon())

def recall(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1-y_true) * (1-y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1-y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * ((p * r) / (p + r + K.epsilon()))

# ============================================
# 3. CONFIG — UPDATE THESE PATHS
# ============================================

MODEL_PATH    = r"D:\compariosn_best_deeplabv3plus_model_model.h5"

# Folder containing the PATCHES you want predictions for
patch_folder  = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches"

# Folder containing the ORIGINAL (un-patched) images — MUST be the same
# folder that was used when the patches were generated. This is needed to
# derive each image's grid shape (rows, cols) exactly like the patch
# generator did.
original_image_folder = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train"

output_folder = r"D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1"

PATCH_SIZE = 512  # must match the patch_size used during patch generation

# If True, the final reconstructed canvas (a multiple of PATCH_SIZE) is
# resized back down to the ORIGINAL image's exact dimensions, undoing the
# resize_to_multiple() upscaling from the patch-generation step.
RESIZE_TO_ORIGINAL = True

# Patch filename pattern, e.g. "846_1_patch000.png" -> base="846_1", idx=0
PATTERN = re.compile(r"^(?P<base>.+)_patch(?P<idx>\d{3})$", re.IGNORECASE)

ORIG_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]

# ============================================
# 4. LOAD MODEL
# ============================================

print("Recreating model architecture...")
model = deeplabv3plus_model(input_size=(PATCH_SIZE, PATCH_SIZE, 3))

print("Loading weights from .h5 file...")
model.load_weights(MODEL_PATH)

print("Compiling model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_focal_dice_loss,
    metrics=[dice_coefficient, iou_score, binary_accuracy,
             precision, recall, specificity, f1_score]
)
print("✅ Model loaded successfully!")

# ============================================
# 5. PREDICTION (in-memory, no intermediate files)
# ============================================

def predict_white_bg_patch(image_path):
    """Runs the model on one 512x512 patch and returns the white-bg masked
    result (RGB uint8, PATCH_SIZE x PATCH_SIZE) — same resolution the model
    operates at, so it drops straight into the reconstruction grid."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (PATCH_SIZE, PATCH_SIZE))
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    pred = model.predict(img_batch, verbose=0)[0]
    pred_binary = (pred > 0.5).astype(np.uint8) * 255
    pred_binary = pred_binary.squeeze()  # (PATCH_SIZE, PATCH_SIZE)

    binary_mask = (pred_binary > 127).astype(np.uint8)
    mask_3ch = np.stack([binary_mask] * 3, axis=-1)
    result = img_resized * mask_3ch
    result = np.where(mask_3ch == 0, 255, result)  # white background

    return result.astype(np.uint8)

# ============================================
# 6. GROUP PATCHES BY BASE IMAGE ID
# ============================================

def group_patches(folder):
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff', '*.tif']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, ext)))
        files.extend(glob.glob(os.path.join(folder, ext.upper())))
    files = list(set(files))

    groups = defaultdict(dict)
    for f in files:
        stem = Path(f).stem
        m = PATTERN.match(stem)
        if not m:
            print(f"  ⚠️  Skipping (doesn't match <base>_patch<NNN> pattern): {Path(f).name}")
            continue
        base = m.group("base")
        idx = int(m.group("idx"))
        groups[base][idx] = f

    return groups

# ============================================
# 7. DERIVE GRID SHAPE FROM THE ORIGINAL IMAGE
#    (mirrors resize_to_multiple() from the patch-generation script)
# ============================================

def find_original_image(base_id):
    for ext in ORIG_EXTENSIONS:
        candidate = os.path.join(original_image_folder, base_id + ext)
        if os.path.exists(candidate):
            return candidate
    return None

def grid_shape_for_image(orig_path, patch_size=PATCH_SIZE):
    img = cv2.imread(orig_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read original image: {orig_path}")
    h, w = img.shape[:2]
    new_h = math.ceil(h / patch_size) * patch_size
    new_w = math.ceil(w / patch_size) * patch_size
    rows = new_h // patch_size
    cols = new_w // patch_size
    return rows, cols, (h, w)

# ============================================
# 8. RECONSTRUCT USING THE ROW-MAJOR GRID
# ============================================

def reconstruct_from_patches(patch_dict, base_id):
    orig_path = find_original_image(base_id)
    if orig_path is None:
        print(f"  ❌ {base_id}: original image not found in {original_image_folder} — cannot determine grid shape, skipping")
        return None

    rows, cols, (orig_h, orig_w) = grid_shape_for_image(orig_path)
    expected_count = rows * cols

    present = sorted(patch_dict.keys())
    missing = [i for i in range(expected_count) if i not in patch_dict]
    if missing:
        print(f"  ⚠️  {base_id}: grid is {rows}x{cols} ({expected_count} patches) but missing index(es) {missing} — those cells will be left white")
    extra = [i for i in present if i >= expected_count]
    if extra:
        print(f"  ⚠️  {base_id}: found patch index(es) {extra} beyond expected grid size {expected_count} — ignoring them")

    canvas = np.full((rows * PATCH_SIZE, cols * PATCH_SIZE, 3), 255, dtype=np.uint8)  # RGB, white

    for idx in present:
        if idx >= expected_count:
            continue
        try:
            pred_patch = predict_white_bg_patch(patch_dict[idx])
        except Exception as e:
            print(f"  ❌ {base_id}: failed predicting patch{idx:03d} ({e}) — leaving that cell white")
            continue
        r = idx // cols
        c = idx % cols
        canvas[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = pred_patch

    if RESIZE_TO_ORIGINAL:
        canvas = cv2.resize(canvas, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    return cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)

# ============================================
# 9. RUN
# ============================================

def main():
    os.makedirs(output_folder, exist_ok=True)

    print(f"\nScanning patches in: {patch_folder}")
    groups = group_patches(patch_folder)

    if not groups:
        print("❌ No matching patch files found.")
        return

    print(f"Found {len(groups)} image groups")
    print("=" * 60)

    success, fail = 0, 0
    for base_id in sorted(groups.keys()):
        print(f"[{base_id}] Predicting {len(groups[base_id])} patch(es) & reconstructing...")
        result = reconstruct_from_patches(groups[base_id], base_id)
        if result is None:
            fail += 1
            continue

        out_path = os.path.join(output_folder, f"{base_id}_reconstructed_white.png")
        cv2.imwrite(out_path, result)
        print(f"  ✅ Saved: {out_path}")
        success += 1

    print("\n" + "=" * 60)
    print("DONE")
    print(f"  ✅ Successful : {success}")
    print(f"  ❌ Failed     : {fail}")
    print(f"  Outputs saved to: {output_folder}/")


if __name__ == "__main__":
    main()

D:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Recreating model architecture...

Loading weights from .h5 file...
Compiling model...
✅ Model loaded successfully!

Scanning patches in: D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches
Found 1198 image groups
[846_1] Predicting 4 patch(es) & reconstructing...


D:\anaconda\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor']. Received: the structure of inputs=*
  warnings.warn(


  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1\846_1_reconstructed_white.png
[846_10] Predicting 4 patch(es) & reconstructing...
  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1\846_10_reconstructed_white.png
[846_100] Predicting 4 patch(es) & reconstructing...
  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1\846_100_reconstructed_white.png
[846_11] Predicting 4 patch(es) & reconstructing...
  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1\846_11_reconstructed_white.png
[846_14] Predicting 4 patch(es) & reconstructing...
  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1\846_14_reconstructed_white.png
[846_17] Predicting 4 patch(es) & reconstructing...
  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-reconstructed1\846_17_reconstructed_white.png
[846_18] Predicting 4 patch(es) & reconstructing...
  ✅ Saved: D:\phd\OneDrive\AMRITA\demo_

In [ ]:
# ============================================
# END-TO-END: DeepLabv3+ Predict -> Reconstruct -> Line-Split
#
# Pipeline (all in memory, no intermediate files written):
#   1. Load model
#   2. For each original image's patches: predict white-bg mask per patch
#   3. Reconstruct the full image using the ACTUAL row-major grid layout
#      from the patch-generation script (rows/cols derived from the
#      ORIGINAL image's dimensions, same as resize_to_multiple() did)
#   4. Split the reconstructed image into individual text lines
#   5. Save ONLY the line crops, to their own output folder, named:
#         <imagename>_line01.png, <imagename>_line02.png, ...
# ============================================

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import numpy as np
import cv2
import os
import re
import math
import glob
from pathlib import Path
from collections import defaultdict
from scipy.ndimage import uniform_filter1d
from scipy.signal import argrelextrema

# ============================================
# 1. MODEL ARCHITECTURE
# ============================================

def depthwise_separable_conv(x, filters, kernel_size=3, strides=1, dilation_rate=1, name_prefix=''):
    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=strides,
        dilation_rate=dilation_rate,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_depthwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_depthwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_depthwise_relu')(x)

    x = Conv2D(
        filters,
        kernel_size=1,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_pointwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_pointwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_pointwise_relu')(x)
    return x

def atrous_spatial_pyramid_pooling(x, output_stride=16):
    b0 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='aspp0')(x)
    b0 = BatchNormalization(name='aspp0_bn')(b0)
    b0 = Activation('relu', name='aspp0_activation')(b0)

    if output_stride == 16:
        atrous_rates = (6, 12, 18)
    elif output_stride == 8:
        atrous_rates = (12, 24, 36)
    else:
        atrous_rates = (6, 12, 18)

    b1 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[0], use_bias=False, name='aspp1')(x)
    b1 = BatchNormalization(name='aspp1_bn')(b1)
    b1 = Activation('relu', name='aspp1_activation')(b1)

    b2 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[1], use_bias=False, name='aspp2')(x)
    b2 = BatchNormalization(name='aspp2_bn')(b2)
    b2 = Activation('relu', name='aspp2_activation')(b2)

    b3 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[2], use_bias=False, name='aspp3')(x)
    b3 = BatchNormalization(name='aspp3_bn')(b3)
    b3 = Activation('relu', name='aspp3_activation')(b3)

    b4 = GlobalAveragePooling2D()(x)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='image_pooling')(b4)
    b4 = BatchNormalization(name='image_pooling_bn')(b4)
    b4 = Activation('relu', name='image_pooling_activation')(b4)

    def resize_to_feature_map(inputs):
        feature_map, pooled_features = inputs
        shape = tf.shape(feature_map)
        target_height = shape[1]
        target_width = shape[2]
        return tf.image.resize(pooled_features, [target_height, target_width])

    b4 = tf.keras.layers.Lambda(resize_to_feature_map)([x, b4])

    x = layers.Concatenate()([b0, b1, b2, b3, b4])

    x = Conv2D(256, (1, 1), padding='same', use_bias=False, name='concat_projection')(x)
    x = BatchNormalization(name='concat_projection_bn')(x)
    x = Activation('relu', name='concat_projection_activation')(x)
    x = Dropout(0.1)(x)

    return x

def deeplabv3plus_model(input_size=(512, 512, 3), output_stride=16):
    inputs = Input(input_size)

    # Entry flow
    x = Conv2D(32, (3, 3), strides=2, padding='same', use_bias=False, name='entry_flow_conv1_1')(inputs)
    x = BatchNormalization(name='entry_flow_conv1_1_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_1_relu')(x)

    x = Conv2D(64, (3, 3), padding='same', use_bias=False, name='entry_flow_conv1_2')(x)
    x = BatchNormalization(name='entry_flow_conv1_2_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_2_relu')(x)

    # Block 1
    residual = Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_1')
    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Block 2
    residual = Conv2D(256, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_1')
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    low_level_features = x

    # Block 3
    residual = Conv2D(728, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_1')
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Middle flow
    for i in range(8):
        residual = x
        prefix = f'middle_flow_unit_{i + 1}'
        x = Activation('relu')(x)
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_1')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_2')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_3')
        x = layers.Add()([x, residual])

    # Exit flow
    residual = Conv2D(1024, (1, 1), strides=1, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='exit_flow_block1_1')
    x = depthwise_separable_conv(x, 1024, name_prefix='exit_flow_block1_2')
    x = layers.Add()([x, residual])

    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_1')
    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_2')
    x = depthwise_separable_conv(x, 2048, name_prefix='exit_flow_block2_3')

    # ASPP
    x = atrous_spatial_pyramid_pooling(x, output_stride)

    # Decoder
    low_level_features = Conv2D(48, (1, 1), padding='same', use_bias=False, name='feature_projection0')(low_level_features)
    low_level_features = BatchNormalization(name='feature_projection0_bn')(low_level_features)
    low_level_features = Activation('relu', name='feature_projection0_relu')(low_level_features)

    def upsample_to_match(inputs):
        encoder_features, low_level_features = inputs
        low_level_shape = tf.shape(low_level_features)
        target_height = low_level_shape[1]
        target_width = low_level_shape[2]
        return tf.image.resize(encoder_features, [target_height, target_width])

    x = tf.keras.layers.Lambda(upsample_to_match)([x, low_level_features])
    x = layers.Concatenate()([x, low_level_features])

    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv0')
    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv1')

    def upsample_to_input_size(x):
        return tf.image.resize(x, [512, 512])

    x = tf.keras.layers.Lambda(upsample_to_input_size)(x)
    outputs = Conv2D(1, kernel_size=1, activation='sigmoid', name='output')(x)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

# ============================================
# 2. LOSS FUNCTIONS & METRICS (needed to compile/load)
# ============================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def combined_focal_dice_loss(y_true, y_pred):
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    alpha = 0.8
    gamma = 2.0
    alpha_t = y_true * alpha + (K.ones_like(y_true) - y_true) * (1 - alpha)
    p_t = y_true * y_pred + (K.ones_like(y_true) - y_true) * (K.ones_like(y_true) - y_pred)
    focal_loss_val = -alpha_t * K.pow((K.ones_like(y_true) - p_t), gamma) * K.log(p_t)
    focal = K.mean(focal_loss_val)
    dice = dice_loss(y_true, y_pred)
    return focal + dice

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def binary_accuracy(y_true, y_pred):
    return K.mean(K.equal(y_true, K.round(y_pred)))

def precision(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    return true_positives / (predicted_positives + K.epsilon())

def recall(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1-y_true) * (1-y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1-y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * ((p * r) / (p + r + K.epsilon()))

# ============================================
# 3. CONFIG — UPDATE THESE PATHS
# ============================================

MODEL_PATH    = r"D:\compariosn_best_deeplabv3plus_model_model.h5"

# Folder containing the PATCHES you want predictions for
patch_folder  = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches"

# Folder containing the ORIGINAL (un-patched) images — MUST be the same
# folder that was used when the patches were generated. Needed to derive
# each image's grid shape (rows, cols), same as resize_to_multiple() did.
original_image_folder = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train"
# ONLY output: extracted line crops go here, named <imagename>_lineNN.png
line_output_folder = r"D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-lines"

PATCH_SIZE = 512  # must match the patch_size used during patch generation

# If True, the reconstructed canvas (a multiple of PATCH_SIZE) is resized
# back down to the ORIGINAL image's exact dimensions before line-splitting.
RESIZE_TO_ORIGINAL = True

# Patch filename pattern, e.g. "846_1_patch000.png" -> base="846_1", idx=0
PATCH_PATTERN = re.compile(r"^(?P<base>.+)_patch(?P<idx>\d{3})$", re.IGNORECASE)

ORIG_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]

# Line-splitting parameters
WHITE_THRESH   = 245   # pixel considered background if gray value >= this
SMOOTH_WINDOW  = 9     # moving-average window for the row profile
MIN_GAP_ORDER  = 8     # local-minimum search window (rows on each side)
MERGE_DIST     = 15    # merge minima closer than this many rows apart
MIN_LINE_ROWS  = 30    # a band with peak foreground pixel-count below this is treated as empty/noise
ROW_PAD        = 3     # extra rows kept above/below each detected line
TRIM_COLUMNS   = True  # tightly crop left/right white margins per line

# ============================================
# 4. LOAD MODEL
# ============================================

print("Recreating model architecture...")
model = deeplabv3plus_model(input_size=(PATCH_SIZE, PATCH_SIZE, 3))

print("Loading weights from .h5 file...")
model.load_weights(MODEL_PATH)

print("Compiling model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_focal_dice_loss,
    metrics=[dice_coefficient, iou_score, binary_accuracy,
             precision, recall, specificity, f1_score]
)
print("✅ Model loaded successfully!")

# ============================================
# 5. PREDICTION (in-memory, per patch)
# ============================================

def predict_white_bg_patch(image_path):
    """Runs the model on one 512x512 patch and returns the white-bg masked
    result (RGB uint8, PATCH_SIZE x PATCH_SIZE)."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (PATCH_SIZE, PATCH_SIZE))
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    pred = model.predict(img_batch, verbose=0)[0]
    pred_binary = (pred > 0.5).astype(np.uint8) * 255
    pred_binary = pred_binary.squeeze()

    binary_mask = (pred_binary > 127).astype(np.uint8)
    mask_3ch = np.stack([binary_mask] * 3, axis=-1)
    result = img_resized * mask_3ch
    result = np.where(mask_3ch == 0, 255, result)  # white background

    return result.astype(np.uint8)

# ============================================
# 6. GROUP PATCHES BY BASE IMAGE ID
# ============================================

def group_patches(folder):
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff', '*.tif']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, ext)))
        files.extend(glob.glob(os.path.join(folder, ext.upper())))
    files = list(set(files))

    groups = defaultdict(dict)
    for f in files:
        stem = Path(f).stem
        m = PATCH_PATTERN.match(stem)
        if not m:
            print(f"  ⚠️  Skipping (doesn't match <base>_patch<NNN> pattern): {Path(f).name}")
            continue
        base = m.group("base")
        idx = int(m.group("idx"))
        groups[base][idx] = f

    return groups

# ============================================
# 7. DERIVE GRID SHAPE FROM THE ORIGINAL IMAGE
#    (mirrors resize_to_multiple() from the patch-generation script)
# ============================================

def find_original_image(base_id):
    for ext in ORIG_EXTENSIONS:
        candidate = os.path.join(original_image_folder, base_id + ext)
        if os.path.exists(candidate):
            return candidate
    return None

def grid_shape_for_image(orig_path, patch_size=PATCH_SIZE):
    img = cv2.imread(orig_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read original image: {orig_path}")
    h, w = img.shape[:2]
    new_h = math.ceil(h / patch_size) * patch_size
    new_w = math.ceil(w / patch_size) * patch_size
    rows = new_h // patch_size
    cols = new_w // patch_size
    return rows, cols, (h, w)

# ============================================
# 8. RECONSTRUCT USING THE ROW-MAJOR GRID (returns array, saves nothing)
# ============================================

def reconstruct_from_patches(patch_dict, base_id):
    orig_path = find_original_image(base_id)
    if orig_path is None:
        print(f"  ❌ {base_id}: original image not found in {original_image_folder} — cannot determine grid shape, skipping")
        return None

    rows, cols, (orig_h, orig_w) = grid_shape_for_image(orig_path)
    expected_count = rows * cols

    present = sorted(patch_dict.keys())
    missing = [i for i in range(expected_count) if i not in patch_dict]
    if missing:
        print(f"  ⚠️  {base_id}: grid is {rows}x{cols} ({expected_count} patches) but missing index(es) {missing} — those cells left white")
    extra = [i for i in present if i >= expected_count]
    if extra:
        print(f"  ⚠️  {base_id}: patch index(es) {extra} beyond expected grid size {expected_count} — ignored")

    canvas = np.full((rows * PATCH_SIZE, cols * PATCH_SIZE, 3), 255, dtype=np.uint8)  # RGB, white

    for idx in present:
        if idx >= expected_count:
            continue
        try:
            pred_patch = predict_white_bg_patch(patch_dict[idx])
        except Exception as e:
            print(f"  ❌ {base_id}: failed predicting patch{idx:03d} ({e}) — leaving that cell white")
            continue
        r = idx // cols
        c = idx % cols
        canvas[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = pred_patch

    if RESIZE_TO_ORIGINAL:
        canvas = cv2.resize(canvas, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)

    return cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)  # BGR, ready for cv2.imwrite

# ============================================
# 9. LINE SPLITTING
# ============================================

def split_into_lines(img):
    """Return a list of (top, bottom) row bounds — one per detected text line."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    fg = (gray < WHITE_THRESH).astype(np.uint8)
    row_sum = fg.sum(axis=1).astype(float)

    if row_sum.max() == 0:
        return []

    smooth = uniform_filter1d(row_sum, size=SMOOTH_WINDOW)

    minima_idx = argrelextrema(smooth, np.less_equal, order=MIN_GAP_ORDER)[0]
    minima_idx = [int(i) for i in minima_idx if smooth[i] < smooth.max() * 0.35]

    merged = []
    for i in minima_idx:
        if merged and i - merged[-1] < MERGE_DIST:
            continue
        merged.append(i)

    h = len(row_sum)
    bounds = sorted(set([0] + merged + [h]))

    lines = []
    for i in range(len(bounds) - 1):
        top, bot = bounds[i], bounds[i + 1]
        if row_sum[top:bot].max() < MIN_LINE_ROWS:
            continue
        lines.append((top, bot))

    return lines


def crop_line(img, top, bot):
    h = img.shape[0]
    top_p = max(0, top - ROW_PAD)
    bot_p = min(h, bot + ROW_PAD)
    crop = img[top_p:bot_p, :]

    if TRIM_COLUMNS:
        crop_gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        col_fg = (crop_gray < WHITE_THRESH).any(axis=0)
        if col_fg.any():
            left = int(np.argmax(col_fg))
            right = len(col_fg) - int(np.argmax(col_fg[::-1]))
            crop = crop[:, left:right]

    return crop

# ============================================
# 10. RUN
# ============================================

def main():
    os.makedirs(line_output_folder, exist_ok=True)

    print(f"\nScanning patches in: {patch_folder}")
    groups = group_patches(patch_folder)

    if not groups:
        print("❌ No matching patch files found.")
        return

    print(f"Found {len(groups)} image groups")
    print("=" * 60)

    images_ok, images_fail, total_lines = 0, 0, 0

    for base_id in sorted(groups.keys()):
        print(f"[{base_id}] Predicting {len(groups[base_id])} patch(es), reconstructing, splitting into lines...")

        reconstructed = reconstruct_from_patches(groups[base_id], base_id)
        if reconstructed is None:
            images_fail += 1
            continue

        lines = split_into_lines(reconstructed)
        if not lines:
            print(f"  ⚠️  {base_id}: no lines detected")
            images_fail += 1
            continue

        for n, (top, bot) in enumerate(lines, start=1):
            crop = crop_line(reconstructed, top, bot)
            out_path = os.path.join(line_output_folder, f"{base_id}_line{n:02d}.png")
            cv2.imwrite(out_path, crop)

        print(f"  ✅ {base_id}: {len(lines)} line(s) saved")
        images_ok += 1
        total_lines += len(lines)

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print(f"  Images processed OK : {images_ok}")
    print(f"  Images failed       : {images_fail}")
    print(f"  Total lines saved   : {total_lines}")
    print(f"  Outputs saved to    : {line_output_folder}/")


if __name__ == "__main__":
    main()

D:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Recreating model architecture...

Loading weights from .h5 file...
Compiling model...
✅ Model loaded successfully!

Scanning patches in: D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches
Found 1198 image groups
[846_1] Predicting 4 patch(es), reconstructing, splitting into lines...


D:\anaconda\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor']. Received: the structure of inputs=*
  warnings.warn(


  ✅ 846_1: 7 line(s) saved
[846_10] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_10: 7 line(s) saved
[846_100] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_100: 4 line(s) saved
[846_11] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_11: 7 line(s) saved
[846_14] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_14: 4 line(s) saved
[846_17] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_17: 8 line(s) saved
[846_18] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_18: 5 line(s) saved
[846_19] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_19: 8 line(s) saved
[846_21] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_21: 7 line(s) saved
[846_24] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_24: 7 line(s) saved
[846_25] Predicting 4 patch(es), reconstructing, splitting int

In [ ]:
# ============================================
# END-TO-END: DeepLabv3+ Predict -> Reconstruct -> Line-Split
#
# Pipeline (all in memory, no intermediate files written):
#   1. Load model
#   2. Predict a binary mask for every patch of an image
#   3. Compute that image's AVERAGE foreground (leaf) color from all its
#      masked-in pixels, and use it to fill the background instead of
#      white/black — so line crops blend with the manuscript instead of
#      having stark white cutouts around them.
#   4. Reconstruct the full image using the ACTUAL row-major grid layout
#      from the patch-generation script (rows/cols derived from the
#      ORIGINAL image's dimensions, same as resize_to_multiple() did)
#   5. Split into individual text lines — using the underlying MASK
#      (not pixel color) to find line boundaries, so detection stays
#      correct even though the background is no longer plain white
#   6. Save ONLY the line crops, named <imagename>_lineNN.png
# ============================================

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import numpy as np
import cv2
import os
import re
import math
import glob
from pathlib import Path
from collections import defaultdict
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

# ============================================
# 1. MODEL ARCHITECTURE
# ============================================

def depthwise_separable_conv(x, filters, kernel_size=3, strides=1, dilation_rate=1, name_prefix=''):
    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=strides,
        dilation_rate=dilation_rate,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_depthwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_depthwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_depthwise_relu')(x)

    x = Conv2D(
        filters,
        kernel_size=1,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_pointwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_pointwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_pointwise_relu')(x)
    return x

def atrous_spatial_pyramid_pooling(x, output_stride=16):
    b0 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='aspp0')(x)
    b0 = BatchNormalization(name='aspp0_bn')(b0)
    b0 = Activation('relu', name='aspp0_activation')(b0)

    if output_stride == 16:
        atrous_rates = (6, 12, 18)
    elif output_stride == 8:
        atrous_rates = (12, 24, 36)
    else:
        atrous_rates = (6, 12, 18)

    b1 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[0], use_bias=False, name='aspp1')(x)
    b1 = BatchNormalization(name='aspp1_bn')(b1)
    b1 = Activation('relu', name='aspp1_activation')(b1)

    b2 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[1], use_bias=False, name='aspp2')(x)
    b2 = BatchNormalization(name='aspp2_bn')(b2)
    b2 = Activation('relu', name='aspp2_activation')(b2)

    b3 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[2], use_bias=False, name='aspp3')(x)
    b3 = BatchNormalization(name='aspp3_bn')(b3)
    b3 = Activation('relu', name='aspp3_activation')(b3)

    b4 = GlobalAveragePooling2D()(x)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='image_pooling')(b4)
    b4 = BatchNormalization(name='image_pooling_bn')(b4)
    b4 = Activation('relu', name='image_pooling_activation')(b4)

    def resize_to_feature_map(inputs):
        feature_map, pooled_features = inputs
        shape = tf.shape(feature_map)
        target_height = shape[1]
        target_width = shape[2]
        return tf.image.resize(pooled_features, [target_height, target_width])

    b4 = tf.keras.layers.Lambda(resize_to_feature_map)([x, b4])

    x = layers.Concatenate()([b0, b1, b2, b3, b4])

    x = Conv2D(256, (1, 1), padding='same', use_bias=False, name='concat_projection')(x)
    x = BatchNormalization(name='concat_projection_bn')(x)
    x = Activation('relu', name='concat_projection_activation')(x)
    x = Dropout(0.1)(x)

    return x

def deeplabv3plus_model(input_size=(512, 512, 3), output_stride=16):
    inputs = Input(input_size)

    # Entry flow
    x = Conv2D(32, (3, 3), strides=2, padding='same', use_bias=False, name='entry_flow_conv1_1')(inputs)
    x = BatchNormalization(name='entry_flow_conv1_1_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_1_relu')(x)

    x = Conv2D(64, (3, 3), padding='same', use_bias=False, name='entry_flow_conv1_2')(x)
    x = BatchNormalization(name='entry_flow_conv1_2_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_2_relu')(x)

    # Block 1
    residual = Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_1')
    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Block 2
    residual = Conv2D(256, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_1')
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    low_level_features = x

    # Block 3
    residual = Conv2D(728, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_1')
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Middle flow
    for i in range(8):
        residual = x
        prefix = f'middle_flow_unit_{i + 1}'
        x = Activation('relu')(x)
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_1')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_2')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_3')
        x = layers.Add()([x, residual])

    # Exit flow
    residual = Conv2D(1024, (1, 1), strides=1, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='exit_flow_block1_1')
    x = depthwise_separable_conv(x, 1024, name_prefix='exit_flow_block1_2')
    x = layers.Add()([x, residual])

    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_1')
    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_2')
    x = depthwise_separable_conv(x, 2048, name_prefix='exit_flow_block2_3')

    # ASPP
    x = atrous_spatial_pyramid_pooling(x, output_stride)

    # Decoder
    low_level_features = Conv2D(48, (1, 1), padding='same', use_bias=False, name='feature_projection0')(low_level_features)
    low_level_features = BatchNormalization(name='feature_projection0_bn')(low_level_features)
    low_level_features = Activation('relu', name='feature_projection0_relu')(low_level_features)

    def upsample_to_match(inputs):
        encoder_features, low_level_features = inputs
        low_level_shape = tf.shape(low_level_features)
        target_height = low_level_shape[1]
        target_width = low_level_shape[2]
        return tf.image.resize(encoder_features, [target_height, target_width])

    x = tf.keras.layers.Lambda(upsample_to_match)([x, low_level_features])
    x = layers.Concatenate()([x, low_level_features])

    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv0')
    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv1')

    def upsample_to_input_size(x):
        return tf.image.resize(x, [512, 512])

    x = tf.keras.layers.Lambda(upsample_to_input_size)(x)
    outputs = Conv2D(1, kernel_size=1, activation='sigmoid', name='output')(x)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

# ============================================
# 2. LOSS FUNCTIONS & METRICS (needed to compile/load)
# ============================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def combined_focal_dice_loss(y_true, y_pred):
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    alpha = 0.8
    gamma = 2.0
    alpha_t = y_true * alpha + (K.ones_like(y_true) - y_true) * (1 - alpha)
    p_t = y_true * y_pred + (K.ones_like(y_true) - y_true) * (K.ones_like(y_true) - y_pred)
    focal_loss_val = -alpha_t * K.pow((K.ones_like(y_true) - p_t), gamma) * K.log(p_t)
    focal = K.mean(focal_loss_val)
    dice = dice_loss(y_true, y_pred)
    return focal + dice

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def binary_accuracy(y_true, y_pred):
    return K.mean(K.equal(y_true, K.round(y_pred)))

def precision(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    return true_positives / (predicted_positives + K.epsilon())

def recall(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1-y_true) * (1-y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1-y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * ((p * r) / (p + r + K.epsilon()))

# ============================================
# 3. CONFIG — UPDATE THESE PATHS
# ============================================

MODEL_PATH    = r"D:\compariosn_best_deeplabv3plus_model_model.h5"

# Folder containing the PATCHES you want predictions for
patch_folder  = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches"

# Folder containing the ORIGINAL (un-patched) images — MUST be the same
# folder that was used when the patches were generated. Needed to derive
# each image's grid shape (rows, cols), same as resize_to_multiple() did.
original_image_folder = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train"

# ONLY output: extracted line crops go here, named <imagename>_lineNN.png
line_output_folder = r"D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-lines"

PATCH_SIZE = 512  # must match the patch_size used during patch generation

# If True, the reconstructed canvas (a multiple of PATCH_SIZE) is resized
# back down to the ORIGINAL image's exact dimensions before line-splitting.
RESIZE_TO_ORIGINAL = True

# Patch filename pattern, e.g. "846_1_patch000.png" -> base="846_1", idx=0
PATCH_PATTERN = re.compile(r"^(?P<base>.+)_patch(?P<idx>\d{3})$", re.IGNORECASE)

ORIG_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]

# Line-splitting parameters (now based on the predicted MASK, not pixel color).
# Lines are found as PEAKS in the row profile (each line's densest row),
# then split at the VALLEY between consecutive peaks. This works even when
# adjacent lines touch and the row profile never dips near zero between
# them — unlike a fixed-threshold valley search, which fails on tightly
# packed lines (the gap is real but shallow, not absolute).
SMOOTH_WINDOW    = 9      # moving-average window for the row profile
MIN_PEAK_DIST    = 15     # minimum rows between two line-center peaks (~half the smallest expected line height)
PEAK_PROMINENCE  = 0.08   # fraction of the profile's max height a bump must stick up by to count as its own line
MIN_LINE_ROWS    = 30     # a band with peak mask pixel-count below this is treated as empty/noise
ROW_PAD          = 3      # extra rows kept above/below each detected line
TRIM_COLUMNS     = True   # tightly crop left/right margins per line, based on the mask

# ============================================
# 4. LOAD MODEL
# ============================================

print("Recreating model architecture...")
model = deeplabv3plus_model(input_size=(PATCH_SIZE, PATCH_SIZE, 3))

print("Loading weights from .h5 file...")
model.load_weights(MODEL_PATH)

print("Compiling model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_focal_dice_loss,
    metrics=[dice_coefficient, iou_score, binary_accuracy,
             precision, recall, specificity, f1_score]
)
print("✅ Model loaded successfully!")

# ============================================
# 5. PREDICTION (in-memory, per patch)
# ============================================

def predict_patch(image_path):
    """Runs the model on one 512x512 patch.
    Returns (patch_rgb uint8 PATCH_SIZExPATCH_SIZEx3, patch_mask uint8 {0,1})."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (PATCH_SIZE, PATCH_SIZE))
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    pred = model.predict(img_batch, verbose=0)[0]
    mask = (pred.squeeze() > 0.5).astype(np.uint8)  # 1 = leaf/text content, 0 = background

    return img_resized, mask

# ============================================
# 6. GROUP PATCHES BY BASE IMAGE ID
# ============================================

def group_patches(folder):
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff', '*.tif']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, ext)))
        files.extend(glob.glob(os.path.join(folder, ext.upper())))
    files = list(set(files))

    groups = defaultdict(dict)
    for f in files:
        stem = Path(f).stem
        m = PATCH_PATTERN.match(stem)
        if not m:
            print(f"  ⚠️  Skipping (doesn't match <base>_patch<NNN> pattern): {Path(f).name}")
            continue
        base = m.group("base")
        idx = int(m.group("idx"))
        groups[base][idx] = f

    return groups

# ============================================
# 7. DERIVE GRID SHAPE FROM THE ORIGINAL IMAGE
#    (mirrors resize_to_multiple() from the patch-generation script)
# ============================================

def find_original_image(base_id):
    for ext in ORIG_EXTENSIONS:
        candidate = os.path.join(original_image_folder, base_id + ext)
        if os.path.exists(candidate):
            return candidate
    return None

def grid_shape_for_image(orig_path, patch_size=PATCH_SIZE):
    img = cv2.imread(orig_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read original image: {orig_path}")
    h, w = img.shape[:2]
    new_h = math.ceil(h / patch_size) * patch_size
    new_w = math.ceil(w / patch_size) * patch_size
    rows = new_h // patch_size
    cols = new_w // patch_size
    return rows, cols, (h, w)

# ============================================
# 8. RECONSTRUCT: predict all patches, derive average leaf color,
#    build the RGB canvas (avg-color background) AND a mask canvas
#    (used later purely for line-boundary detection).
# ============================================

def reconstruct_from_patches(patch_dict, base_id):
    orig_path = find_original_image(base_id)
    if orig_path is None:
        print(f"  ❌ {base_id}: original image not found in {original_image_folder} — cannot determine grid shape, skipping")
        return None, None

    rows, cols, (orig_h, orig_w) = grid_shape_for_image(orig_path)
    expected_count = rows * cols

    present = sorted(patch_dict.keys())
    missing = [i for i in range(expected_count) if i not in patch_dict]
    if missing:
        print(f"  ⚠️  {base_id}: grid is {rows}x{cols} ({expected_count} patches) but missing index(es) {missing}")
    extra = [i for i in present if i >= expected_count]
    if extra:
        print(f"  ⚠️  {base_id}: patch index(es) {extra} beyond expected grid size {expected_count} — ignored")

    # --- Phase 1: predict every patch, keep results in memory, accumulate
    #     the average color of all masked-in (foreground) pixels ---
    patch_results = {}
    fg_sum = np.zeros(3, dtype=np.float64)
    fg_count = 0

    for idx in present:
        if idx >= expected_count:
            continue
        try:
            patch_rgb, patch_mask = predict_patch(patch_dict[idx])
        except Exception as e:
            print(f"  ❌ {base_id}: failed predicting patch{idx:03d} ({e})")
            continue
        patch_results[idx] = (patch_rgb, patch_mask)
        fg_pixels = patch_rgb[patch_mask == 1]
        if fg_pixels.size > 0:
            fg_sum += fg_pixels.sum(axis=0)
            fg_count += fg_pixels.shape[0]

    if fg_count == 0:
        print(f"  ⚠️  {base_id}: no foreground pixels detected in any patch — falling back to white background")
        avg_color = np.array([255, 255, 255], dtype=np.uint8)
    else:
        avg_color = (fg_sum / fg_count).astype(np.uint8)

    # --- Phase 2: build the RGB canvas (avg-color background) and the
    #     mask canvas (for line detection) ---
    rgb_canvas = np.empty((rows * PATCH_SIZE, cols * PATCH_SIZE, 3), dtype=np.uint8)
    rgb_canvas[:] = avg_color
    mask_canvas = np.zeros((rows * PATCH_SIZE, cols * PATCH_SIZE), dtype=np.uint8)

    for idx, (patch_rgb, patch_mask) in patch_results.items():
        r = idx // cols
        c = idx % cols
        masked_patch = np.where(patch_mask[..., None] == 1, patch_rgb, avg_color)
        rgb_canvas[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = masked_patch
        mask_canvas[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = patch_mask

    if RESIZE_TO_ORIGINAL:
        rgb_canvas = cv2.resize(rgb_canvas, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
        mask_canvas = cv2.resize(mask_canvas, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    bgr_canvas = cv2.cvtColor(rgb_canvas, cv2.COLOR_RGB2BGR)
    return bgr_canvas, mask_canvas

# ============================================
# 9. LINE SPLITTING — driven by the MASK, not pixel color, so it stays
#    correct regardless of what the background is filled with.
# ============================================

def split_into_lines(mask):
    """Return a list of (top, bottom) row bounds — one per detected text line.

    Finds each line as a PEAK in the row profile (its densest row), then
    cuts at the minimum point (valley) between each pair of consecutive
    peaks. This finds the right number of lines even when neighboring
    lines touch and there's no near-zero gap between them, because it
    only needs a RELATIVE dip between two peaks, not an absolute one."""
    row_sum = mask.sum(axis=1).astype(float)

    if row_sum.max() == 0:
        return []

    smooth = uniform_filter1d(row_sum, size=SMOOTH_WINDOW)

    peaks, _ = find_peaks(
        smooth,
        distance=MIN_PEAK_DIST,
        prominence=smooth.max() * PEAK_PROMINENCE,
    )

    if len(peaks) == 0:
        # Fallback: no clear peak structure — treat as a single line if
        # there's any foreground at all.
        return [(0, len(row_sum))] if row_sum.max() >= MIN_LINE_ROWS else []

    h = len(row_sum)
    bounds = [0]
    for i in range(len(peaks) - 1):
        a, b = peaks[i], peaks[i + 1]
        valley = a + int(np.argmin(smooth[a:b + 1]))
        bounds.append(valley)
    bounds.append(h)

    lines = []
    for i in range(len(bounds) - 1):
        top, bot = bounds[i], bounds[i + 1]
        if row_sum[top:bot].max() < MIN_LINE_ROWS:
            continue
        lines.append((top, bot))

    return lines


def crop_line(img, mask, top, bot):
    h = img.shape[0]
    top_p = max(0, top - ROW_PAD)
    bot_p = min(h, bot + ROW_PAD)
    crop = img[top_p:bot_p, :]
    mask_band = mask[top_p:bot_p, :]

    if TRIM_COLUMNS:
        col_fg = mask_band.any(axis=0)
        if col_fg.any():
            left = int(np.argmax(col_fg))
            right = len(col_fg) - int(np.argmax(col_fg[::-1]))
            crop = crop[:, left:right]

    return crop

# ============================================
# 10. RUN
# ============================================

def main():
    os.makedirs(line_output_folder, exist_ok=True)

    print(f"\nScanning patches in: {patch_folder}")
    groups = group_patches(patch_folder)

    if not groups:
        print("❌ No matching patch files found.")
        return

    print(f"Found {len(groups)} image groups")
    print("=" * 60)

    images_ok, images_fail, total_lines = 0, 0, 0

    for base_id in sorted(groups.keys()):
        print(f"[{base_id}] Predicting {len(groups[base_id])} patch(es), reconstructing, splitting into lines...")

        reconstructed, mask = reconstruct_from_patches(groups[base_id], base_id)
        if reconstructed is None:
            images_fail += 1
            continue

        lines = split_into_lines(mask)
        if not lines:
            print(f"  ⚠️  {base_id}: no lines detected")
            images_fail += 1
            continue

        for n, (top, bot) in enumerate(lines, start=1):
            crop = crop_line(reconstructed, mask, top, bot)
            out_path = os.path.join(line_output_folder, f"{base_id}_line{n:02d}.png")
            cv2.imwrite(out_path, crop)

        print(f"  ✅ {base_id}: {len(lines)} line(s) saved")
        images_ok += 1
        total_lines += len(lines)

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print(f"  Images processed OK : {images_ok}")
    print(f"  Images failed       : {images_fail}")
    print(f"  Total lines saved   : {total_lines}")
    print(f"  Outputs saved to    : {line_output_folder}/")


if __name__ == "__main__":
    main()

D:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Recreating model architecture...

Loading weights from .h5 file...
Compiling model...
✅ Model loaded successfully!

Scanning patches in: D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches
Found 1198 image groups
[846_1] Predicting 4 patch(es), reconstructing, splitting into lines...


D:\anaconda\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor']. Received: the structure of inputs=*
  warnings.warn(


  ✅ 846_1: 7 line(s) saved
[846_10] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_10: 7 line(s) saved
[846_100] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_100: 4 line(s) saved
[846_11] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_11: 7 line(s) saved
[846_14] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_14: 7 line(s) saved
[846_17] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_17: 8 line(s) saved
[846_18] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_18: 7 line(s) saved
[846_19] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_19: 7 line(s) saved
[846_21] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_21: 7 line(s) saved
[846_24] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_24: 7 line(s) saved
[846_25] Predicting 4 patch(es), reconstructing, splitting int

In [1]:
# ============================================
# END-TO-END: DeepLabv3+ Predict -> Reconstruct -> Line-Split
#
# Pipeline (all in memory, no intermediate files written):
#   1. Load model
#   2. Predict a binary mask for every patch of an image
#   3. Compute that image's AVERAGE foreground (leaf) color from all its
#      masked-in pixels, and use it to fill the background instead of
#      white/black — so line crops blend with the manuscript instead of
#      having stark white cutouts around them.
#   4. Reconstruct the full image using the ACTUAL row-major grid layout
#      from the patch-generation script (rows/cols derived from the
#      ORIGINAL image's dimensions, same as resize_to_multiple() did)
#   5. Split into individual text lines — using the underlying MASK
#      (not pixel color) to find line boundaries, so detection stays
#      correct even though the background is no longer plain white
#   6. Save ONLY the line crops, named <imagename>_lineNN.png
# ============================================

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import numpy as np
import cv2
import os
import re
import math
import glob
from pathlib import Path
from collections import defaultdict
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

# ============================================
# 1. MODEL ARCHITECTURE
# ============================================

def depthwise_separable_conv(x, filters, kernel_size=3, strides=1, dilation_rate=1, name_prefix=''):
    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=strides,
        dilation_rate=dilation_rate,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_depthwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_depthwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_depthwise_relu')(x)

    x = Conv2D(
        filters,
        kernel_size=1,
        padding='same',
        use_bias=False,
        name=f'{name_prefix}_pointwise'
    )(x)
    x = BatchNormalization(name=f'{name_prefix}_pointwise_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_pointwise_relu')(x)
    return x

def atrous_spatial_pyramid_pooling(x, output_stride=16):
    b0 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='aspp0')(x)
    b0 = BatchNormalization(name='aspp0_bn')(b0)
    b0 = Activation('relu', name='aspp0_activation')(b0)

    if output_stride == 16:
        atrous_rates = (6, 12, 18)
    elif output_stride == 8:
        atrous_rates = (12, 24, 36)
    else:
        atrous_rates = (6, 12, 18)

    b1 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[0], use_bias=False, name='aspp1')(x)
    b1 = BatchNormalization(name='aspp1_bn')(b1)
    b1 = Activation('relu', name='aspp1_activation')(b1)

    b2 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[1], use_bias=False, name='aspp2')(x)
    b2 = BatchNormalization(name='aspp2_bn')(b2)
    b2 = Activation('relu', name='aspp2_activation')(b2)

    b3 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[2], use_bias=False, name='aspp3')(x)
    b3 = BatchNormalization(name='aspp3_bn')(b3)
    b3 = Activation('relu', name='aspp3_activation')(b3)

    b4 = GlobalAveragePooling2D()(x)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, 1))(b4)
    b4 = Conv2D(256, (1, 1), padding='same', use_bias=False, name='image_pooling')(b4)
    b4 = BatchNormalization(name='image_pooling_bn')(b4)
    b4 = Activation('relu', name='image_pooling_activation')(b4)

    def resize_to_feature_map(inputs):
        feature_map, pooled_features = inputs
        shape = tf.shape(feature_map)
        target_height = shape[1]
        target_width = shape[2]
        return tf.image.resize(pooled_features, [target_height, target_width])

    b4 = tf.keras.layers.Lambda(resize_to_feature_map)([x, b4])

    x = layers.Concatenate()([b0, b1, b2, b3, b4])

    x = Conv2D(256, (1, 1), padding='same', use_bias=False, name='concat_projection')(x)
    x = BatchNormalization(name='concat_projection_bn')(x)
    x = Activation('relu', name='concat_projection_activation')(x)
    x = Dropout(0.1)(x)

    return x

def deeplabv3plus_model(input_size=(512, 512, 3), output_stride=16):
    inputs = Input(input_size)

    # Entry flow
    x = Conv2D(32, (3, 3), strides=2, padding='same', use_bias=False, name='entry_flow_conv1_1')(inputs)
    x = BatchNormalization(name='entry_flow_conv1_1_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_1_relu')(x)

    x = Conv2D(64, (3, 3), padding='same', use_bias=False, name='entry_flow_conv1_2')(x)
    x = BatchNormalization(name='entry_flow_conv1_2_bn')(x)
    x = Activation('relu', name='entry_flow_conv1_2_relu')(x)

    # Block 1
    residual = Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_1')
    x = depthwise_separable_conv(x, 128, name_prefix='entry_flow_block1_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Block 2
    residual = Conv2D(256, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_1')
    x = depthwise_separable_conv(x, 256, name_prefix='entry_flow_block2_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    low_level_features = x

    # Block 3
    residual = Conv2D(728, (1, 1), strides=2, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_1')
    x = depthwise_separable_conv(x, 728, name_prefix='entry_flow_block3_2')
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)
    x = layers.Add()([x, residual])

    # Middle flow
    for i in range(8):
        residual = x
        prefix = f'middle_flow_unit_{i + 1}'
        x = Activation('relu')(x)
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_1')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_2')
        x = depthwise_separable_conv(x, 728, name_prefix=f'{prefix}_3')
        x = layers.Add()([x, residual])

    # Exit flow
    residual = Conv2D(1024, (1, 1), strides=1, padding='same', use_bias=False)(x)
    residual = BatchNormalization()(residual)

    x = Activation('relu')(x)
    x = depthwise_separable_conv(x, 728, name_prefix='exit_flow_block1_1')
    x = depthwise_separable_conv(x, 1024, name_prefix='exit_flow_block1_2')
    x = layers.Add()([x, residual])

    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_1')
    x = depthwise_separable_conv(x, 1536, name_prefix='exit_flow_block2_2')
    x = depthwise_separable_conv(x, 2048, name_prefix='exit_flow_block2_3')

    # ASPP
    x = atrous_spatial_pyramid_pooling(x, output_stride)

    # Decoder
    low_level_features = Conv2D(48, (1, 1), padding='same', use_bias=False, name='feature_projection0')(low_level_features)
    low_level_features = BatchNormalization(name='feature_projection0_bn')(low_level_features)
    low_level_features = Activation('relu', name='feature_projection0_relu')(low_level_features)

    def upsample_to_match(inputs):
        encoder_features, low_level_features = inputs
        low_level_shape = tf.shape(low_level_features)
        target_height = low_level_shape[1]
        target_width = low_level_shape[2]
        return tf.image.resize(encoder_features, [target_height, target_width])

    x = tf.keras.layers.Lambda(upsample_to_match)([x, low_level_features])
    x = layers.Concatenate()([x, low_level_features])

    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv0')
    x = depthwise_separable_conv(x, 256, name_prefix='decoder_conv1')

    def upsample_to_input_size(x):
        return tf.image.resize(x, [512, 512])

    x = tf.keras.layers.Lambda(upsample_to_input_size)(x)
    outputs = Conv2D(1, kernel_size=1, activation='sigmoid', name='output')(x)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

# ============================================
# 2. LOSS FUNCTIONS & METRICS (needed to compile/load)
# ============================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

def combined_focal_dice_loss(y_true, y_pred):
    epsilon = K.epsilon()
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    alpha = 0.8
    gamma = 2.0
    alpha_t = y_true * alpha + (K.ones_like(y_true) - y_true) * (1 - alpha)
    p_t = y_true * y_pred + (K.ones_like(y_true) - y_true) * (K.ones_like(y_true) - y_pred)
    focal_loss_val = -alpha_t * K.pow((K.ones_like(y_true) - p_t), gamma) * K.log(p_t)
    focal = K.mean(focal_loss_val)
    dice = dice_loss(y_true, y_pred)
    return focal + dice

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

def binary_accuracy(y_true, y_pred):
    return K.mean(K.equal(y_true, K.round(y_pred)))

def precision(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    return true_positives / (predicted_positives + K.epsilon())

def recall(y_true, y_pred):
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    return true_positives / (possible_positives + K.epsilon())

def specificity(y_true, y_pred):
    true_negatives = K.sum(K.round(K.clip((1-y_true) * (1-y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1-y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * ((p * r) / (p + r + K.epsilon()))

# ============================================
# 3. CONFIG — UPDATE THESE PATHS
# ============================================

MODEL_PATH    = r"D:\compariosn_best_deeplabv3plus_model_model.h5"

# Folder containing the PATCHES you want predictions for
patch_folder  = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches"

# Folder containing the ORIGINAL (un-patched) images — MUST be the same
# folder that was used when the patches were generated. Needed to derive
# each image's grid shape (rows, cols), same as resize_to_multiple() did.
original_image_folder = r"D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train"

# ONLY output: extracted line crops go here, named <imagename>_lineNN.png
line_output_folder = r"D:\phd\OneDrive\AMRITA\demo_02-09-2026\image-train-lines"

PATCH_SIZE = 512  # must match the patch_size used during patch generation

# If True, the reconstructed canvas (a multiple of PATCH_SIZE) is resized
# back down to the ORIGINAL image's exact dimensions before line-splitting.
RESIZE_TO_ORIGINAL = True

# Patch filename pattern, e.g. "846_1_patch000.png" -> base="846_1", idx=0
PATCH_PATTERN = re.compile(r"^(?P<base>.+)_patch(?P<idx>\d{3})$", re.IGNORECASE)

ORIG_EXTENSIONS = [".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"]

# Line-splitting parameters (now based on the predicted MASK, not pixel color).
# Lines are found as PEAKS in the row profile (each line's densest row),
# then split at the VALLEY between consecutive peaks. This works even when
# adjacent lines touch and the row profile never dips near zero between
# them — unlike a fixed-threshold valley search, which fails on tightly
# packed lines (the gap is real but shallow, not absolute).
SMOOTH_WINDOW    = 9      # moving-average window for the row profile
MIN_PEAK_DIST    = 15     # minimum rows between two line-center peaks (~half the smallest expected line height)
PEAK_PROMINENCE  = 0.08   # fraction of the profile's max height a bump must stick up by to count as its own line
MIN_LINE_ROWS    = 30     # a band with peak mask pixel-count below this is treated as empty/noise
ROW_PAD          = 3      # extra rows kept above/below each detected line
TRIM_COLUMNS     = True   # tightly crop left/right margins per line, based on the mask

# ============================================
# 4. LOAD MODEL
# ============================================

print("Recreating model architecture...")
model = deeplabv3plus_model(input_size=(PATCH_SIZE, PATCH_SIZE, 3))

print("Loading weights from .h5 file...")
model.load_weights(MODEL_PATH)

print("Compiling model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_focal_dice_loss,
    metrics=[dice_coefficient, iou_score, binary_accuracy,
             precision, recall, specificity, f1_score]
)
print("✅ Model loaded successfully!")

# ============================================
# 5. PREDICTION (in-memory, per patch)
# ============================================

def predict_patch(image_path):
    """Runs the model on one 512x512 patch.
    Returns (patch_rgb uint8 PATCH_SIZExPATCH_SIZEx3, patch_mask uint8 {0,1})."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (PATCH_SIZE, PATCH_SIZE))
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    pred = model.predict(img_batch, verbose=0)[0]
    mask = (pred.squeeze() > 0.5).astype(np.uint8)  # 1 = leaf/text content, 0 = background

    return img_resized, mask

# ============================================
# 6. GROUP PATCHES BY BASE IMAGE ID
# ============================================

def group_patches(folder):
    extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tiff', '*.tif']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, ext)))
        files.extend(glob.glob(os.path.join(folder, ext.upper())))
    files = list(set(files))

    groups = defaultdict(dict)
    for f in files:
        stem = Path(f).stem
        m = PATCH_PATTERN.match(stem)
        if not m:
            print(f"  ⚠️  Skipping (doesn't match <base>_patch<NNN> pattern): {Path(f).name}")
            continue
        base = m.group("base")
        idx = int(m.group("idx"))
        groups[base][idx] = f

    return groups

# ============================================
# 7. DERIVE GRID SHAPE FROM THE ORIGINAL IMAGE
#    (mirrors resize_to_multiple() from the patch-generation script)
# ============================================

def find_original_image(base_id):
    for ext in ORIG_EXTENSIONS:
        candidate = os.path.join(original_image_folder, base_id + ext)
        if os.path.exists(candidate):
            return candidate
    return None

def grid_shape_for_image(orig_path, patch_size=PATCH_SIZE):
    img = cv2.imread(orig_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read original image: {orig_path}")
    h, w = img.shape[:2]
    new_h = math.ceil(h / patch_size) * patch_size
    new_w = math.ceil(w / patch_size) * patch_size
    rows = new_h // patch_size
    cols = new_w // patch_size
    return rows, cols, (h, w)

# ============================================
# 8. RECONSTRUCT: predict all patches, derive average leaf color,
#    build the RGB canvas (avg-color background) AND a mask canvas
#    (used later purely for line-boundary detection).
# ============================================

def reconstruct_from_patches(patch_dict, base_id):
    orig_path = find_original_image(base_id)
    if orig_path is None:
        print(f"  ❌ {base_id}: original image not found in {original_image_folder} — cannot determine grid shape, skipping")
        return None, None

    rows, cols, (orig_h, orig_w) = grid_shape_for_image(orig_path)
    expected_count = rows * cols

    present = sorted(patch_dict.keys())
    missing = [i for i in range(expected_count) if i not in patch_dict]
    if missing:
        print(f"  ⚠️  {base_id}: grid is {rows}x{cols} ({expected_count} patches) but missing index(es) {missing}")
    extra = [i for i in present if i >= expected_count]
    if extra:
        print(f"  ⚠️  {base_id}: patch index(es) {extra} beyond expected grid size {expected_count} — ignored")

    # --- Phase 1: predict every patch, keep results in memory, accumulate
    #     the average color of all masked-in (foreground) pixels ---
    patch_results = {}
    fg_sum = np.zeros(3, dtype=np.float64)
    fg_count = 0

    for idx in present:
        if idx >= expected_count:
            continue
        try:
            patch_rgb, patch_mask = predict_patch(patch_dict[idx])
        except Exception as e:
            print(f"  ❌ {base_id}: failed predicting patch{idx:03d} ({e})")
            continue
        patch_results[idx] = (patch_rgb, patch_mask)
        fg_pixels = patch_rgb[patch_mask == 1]
        if fg_pixels.size > 0:
            fg_sum += fg_pixels.sum(axis=0)
            fg_count += fg_pixels.shape[0]

    if fg_count == 0:
        print(f"  ⚠️  {base_id}: no foreground pixels detected in any patch — falling back to white background")
        avg_color = np.array([255, 255, 255], dtype=np.uint8)
    else:
        avg_color = (fg_sum / fg_count).astype(np.uint8)

    # --- Phase 2: build the RGB canvas (avg-color background) and the
    #     mask canvas (for line detection) ---
    rgb_canvas = np.empty((rows * PATCH_SIZE, cols * PATCH_SIZE, 3), dtype=np.uint8)
    rgb_canvas[:] = avg_color
    mask_canvas = np.zeros((rows * PATCH_SIZE, cols * PATCH_SIZE), dtype=np.uint8)

    for idx, (patch_rgb, patch_mask) in patch_results.items():
        r = idx // cols
        c = idx % cols
        masked_patch = np.where(patch_mask[..., None] == 1, patch_rgb, avg_color)
        rgb_canvas[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = masked_patch
        mask_canvas[r*PATCH_SIZE:(r+1)*PATCH_SIZE, c*PATCH_SIZE:(c+1)*PATCH_SIZE] = patch_mask

    if RESIZE_TO_ORIGINAL:
        rgb_canvas = cv2.resize(rgb_canvas, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
        mask_canvas = cv2.resize(mask_canvas, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

    bgr_canvas = cv2.cvtColor(rgb_canvas, cv2.COLOR_RGB2BGR)
    return bgr_canvas, mask_canvas

# ============================================
# 9. LINE SPLITTING — driven by the MASK, not pixel color, so it stays
#    correct regardless of what the background is filled with.
# ============================================

def _tighten_band(row_sum, top, bot):
    """Shrink a rough (top, bot) band down to the actual extent of
    foreground rows inside it. Rough bands come from valleys (or, for an
    edge/single-peak band, from the image border) and can include a lot
    of empty margin that has no foreground at all — this trims that off
    before padding is added back in crop_line()."""
    band = row_sum[top:bot]
    nonzero = np.where(band > 0)[0]
    if len(nonzero) == 0:
        return top, bot  # nothing to tighten against; leave as-is
    return top + int(nonzero[0]), top + int(nonzero[-1]) + 1


def split_into_lines(mask):
    """Return a list of (top, bottom) row bounds — one per detected text line.

    Finds each line as a PEAK in the row profile (its densest row), then
    cuts at the minimum point (valley) between each pair of consecutive
    peaks. This finds the right number of lines even when neighboring
    lines touch and there's no near-zero gap between them, because it
    only needs a RELATIVE dip between two peaks, not an absolute one.

    The outer edges (before the first peak, after the last peak — or the
    whole image when there's only one peak) default to the image border,
    which can include a lot of empty margin. Each band is tightened down
    to where its foreground actually starts/ends before being returned."""
    row_sum = mask.sum(axis=1).astype(float)

    if row_sum.max() == 0:
        return []

    smooth = uniform_filter1d(row_sum, size=SMOOTH_WINDOW)

    peaks, _ = find_peaks(
        smooth,
        distance=MIN_PEAK_DIST,
        prominence=smooth.max() * PEAK_PROMINENCE,
    )

    if len(peaks) == 0:
        # No clear peak structure — treat as a single line if there's any
        # foreground at all, tightened to its actual extent.
        if row_sum.max() < MIN_LINE_ROWS:
            return []
        return [_tighten_band(row_sum, 0, len(row_sum))]

    h = len(row_sum)
    bounds = [0]
    for i in range(len(peaks) - 1):
        a, b = peaks[i], peaks[i + 1]
        valley = a + int(np.argmin(smooth[a:b + 1]))
        bounds.append(valley)
    bounds.append(h)

    lines = []
    for i in range(len(bounds) - 1):
        top, bot = bounds[i], bounds[i + 1]
        if row_sum[top:bot].max() < MIN_LINE_ROWS:
            continue
        lines.append(_tighten_band(row_sum, top, bot))

    return lines


def crop_line(img, mask, top, bot):
    h = img.shape[0]
    top_p = max(0, top - ROW_PAD)
    bot_p = min(h, bot + ROW_PAD)
    crop = img[top_p:bot_p, :]
    mask_band = mask[top_p:bot_p, :]

    if TRIM_COLUMNS:
        col_fg = mask_band.any(axis=0)
        if col_fg.any():
            left = int(np.argmax(col_fg))
            right = len(col_fg) - int(np.argmax(col_fg[::-1]))
            crop = crop[:, left:right]

    return crop

# ============================================
# 10. RUN
# ============================================

def main():
    os.makedirs(line_output_folder, exist_ok=True)

    print(f"\nScanning patches in: {patch_folder}")
    groups = group_patches(patch_folder)

    if not groups:
        print("❌ No matching patch files found.")
        return

    print(f"Found {len(groups)} image groups")
    print("=" * 60)

    images_ok, images_fail, total_lines = 0, 0, 0

    for base_id in sorted(groups.keys()):
        print(f"[{base_id}] Predicting {len(groups[base_id])} patch(es), reconstructing, splitting into lines...")

        reconstructed, mask = reconstruct_from_patches(groups[base_id], base_id)
        if reconstructed is None:
            images_fail += 1
            continue

        lines = split_into_lines(mask)
        if not lines:
            print(f"  ⚠️  {base_id}: no lines detected")
            images_fail += 1
            continue

        for n, (top, bot) in enumerate(lines, start=1):
            crop = crop_line(reconstructed, mask, top, bot)
            out_path = os.path.join(line_output_folder, f"{base_id}_line{n:02d}.png")
            cv2.imwrite(out_path, crop)

        print(f"  ✅ {base_id}: {len(lines)} line(s) saved")
        images_ok += 1
        total_lines += len(lines)

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETE")
    print(f"  Images processed OK : {images_ok}")
    print(f"  Images failed       : {images_fail}")
    print(f"  Total lines saved   : {total_lines}")
    print(f"  Outputs saved to    : {line_output_folder}/")


if __name__ == "__main__":
    main()

D:\anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Recreating model architecture...

Loading weights from .h5 file...
Compiling model...
✅ Model loaded successfully!

Scanning patches in: D:\phd\OneDrive\AMRITA\line segmentaion\for labelling\qualitywise implementaion\PAPER\LEAF_OCR_line\LeafOCR-Line\demo-linesegdataset\image-train-patches
Found 1198 image groups
[846_1] Predicting 4 patch(es), reconstructing, splitting into lines...


D:\anaconda\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor']. Received: the structure of inputs=*
  warnings.warn(


  ✅ 846_1: 7 line(s) saved
[846_10] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_10: 7 line(s) saved
[846_100] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_100: 4 line(s) saved
[846_11] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_11: 7 line(s) saved
[846_14] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_14: 7 line(s) saved
[846_17] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_17: 8 line(s) saved
[846_18] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_18: 7 line(s) saved
[846_19] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_19: 7 line(s) saved
[846_21] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_21: 7 line(s) saved
[846_24] Predicting 4 patch(es), reconstructing, splitting into lines...
  ✅ 846_24: 7 line(s) saved
[846_25] Predicting 4 patch(es), reconstructing, splitting int